In [ ]:
%load_ext autoreload
%autoreload 2


# Trading Competition Training

Train Nash-DQN and locally linear-quadratic SRE-DQN on the shared trading competition environment.

In [ ]:
from pathlib import Path
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import torch


def find_repo_root(start):
    start = Path(start).resolve()
    for path in [start, *start.parents]:
        if (path / 'AGENTS.md').exists() and (path / 'continuous_action_space').exists():
            return path
    raise RuntimeError('Could not locate repo root')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXPERIMENT_DIR = REPO_ROOT / 'continuous_action_space' / 'trading_competition'
MODEL_ROOT = EXPERIMENT_DIR / 'pt_files'
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

np.set_printoptions(precision=4)
print('Repo root:', REPO_ROOT)
print('Model root:', MODEL_ROOT)
print('Using device:', 'cuda' if torch.cuda.is_available() else 'cpu')


## Imports And Helpers


In [ ]:
from continuous_action_space.trading_competition.experiment_config import (
    NUM_PLAYERS as num_players,
    sim_dict,
    norm_mean,
    norm_std,
    T,
    make_sim_obj,
    seed_everything,
    configure_torch_gpu_math,
    eps_slug,
    seed_slug,
    BASE_SEED,
    MAX_STEPS,
    LLQ_SRE_EPS_LIST,
    SRE_EPS_REG,
    SRE_DELTA_MIN,
    SRE_GAMMA,
    NET_KWARGS,
)
from continuous_action_space.trading_competition.training import run_training_loop
from continuous_action_space.locally_linear_quadratic.NashAgent_lib import NashNN
from continuous_action_space.locally_linear_quadratic.sre_agent import SreNN

configure_torch_gpu_math()
sim_obj = make_sim_obj()


def make_nash_action(agent, eps_b, noise_std):
    del eps_b
    def fn(cur_s, cur_ivt):
        mu = agent.predict_action(cur_s, cur_ivt)[:, 4]
        return mu + torch.randn_like(mu) * noise_std
    return fn


def make_llq_sre_action(agent, eps_b, noise_std):
    def fn(cur_s, cur_ivt):
        mu_sr = agent.compute_sre_action(cur_s, cur_ivt, eps_b)
        return mu_sr + torch.randn_like(mu_sr) * noise_std
    return fn


def make_eps_schedule(eps_0):
    def eps_schedule(k, num_sim):
        if SRE_EPS_DECAY is None:
            return eps_0
        return eps_0 * max(0.0, 1.0 - k / max(SRE_EPS_DECAY, 1))
    return eps_schedule


print('Agents:', num_players, '| T=', sim_dict['T'].item(), '| dt=', sim_dict['dt'].item(), '| impact=', sim_obj.impact)

## Training Configuration


In [ ]:
NASH_SEED = BASE_SEED
LLQ_SRE_TRAIN_SEED = BASE_SEED

NUM_SIM = 10000
RV_MIN = 0.5
RV_MAX = 2.5
EARLY_STOP = True
EARLY_LIM = 2000
MINI_BATCH = 256
SRE_EPS_DECAY = None

TRAIN_NASH = True
TRAIN_LLQ_SRE = True

TRAINING_LOG_KWARGS = dict(
    loss_log_every=1000,
    eval_reward_every=1000,
    eval_episodes=500,
    log_best_updates=False,
    log_weight_saves=False,
)

NASH_RUN_TAG = seed_slug(NASH_SEED)
LLQ_SRE_RUN_TAG = seed_slug(LLQ_SRE_TRAIN_SEED)

NASH_MODEL_DIR = MODEL_ROOT / f'nash_{NASH_RUN_TAG}'
LLQ_SRE_MODEL_DIRS = {
    eps: MODEL_ROOT / f'llq_sre_eps_{eps_slug(eps)}_{LLQ_SRE_RUN_TAG}'
    for eps in LLQ_SRE_EPS_LIST
}

seed_everything(BASE_SEED)
print('NUM_SIM:', NUM_SIM, '| MAX_STEPS:', MAX_STEPS, '| MINI_BATCH:', MINI_BATCH)
print('LLQ Epsilon sweep:', LLQ_SRE_EPS_LIST, '| decay horizon:', SRE_EPS_DECAY)
print('Nash dir:', NASH_MODEL_DIR)

## Train Nash-DQN


In [ ]:
nash_agent = None
nash_loss = None
nash_train_time = None

if TRAIN_NASH:
    NASH_MODEL_DIR.mkdir(parents=True, exist_ok=True)
    seed_everything(NASH_SEED)
    nash_agent = NashNN(**NET_KWARGS)
    start = time.time()
    nash_agent, nash_loss = run_training_loop(
        sim_obj=sim_obj,
        sim_dict=sim_dict,
        max_steps=MAX_STEPS,
        agent=nash_agent,
        make_action_fn=make_nash_action,
        eps_schedule_fn=None,
        num_sim=NUM_SIM,
        norm_mean=norm_mean,
        norm_std=norm_std,
        rv_min=RV_MIN,
        rv_max=RV_MAX,
        early_stop=EARLY_STOP,
        early_lim=EARLY_LIM,
        mini_batch=MINI_BATCH,
        path=str(NASH_MODEL_DIR),
        AN_file_name=str(NASH_MODEL_DIR / 'Action_Net'),
        VN_file_name=str(NASH_MODEL_DIR / 'Value_Net'),
        checkpoint_metadata={'trainer': 'nash_dqn', 'seed': int(NASH_SEED)},
        desc='Nash-DQN',
        **TRAINING_LOG_KWARGS,
    )
    nash_train_time = time.time() - start
    print(f'Nash-DQN training time: {nash_train_time:.1f}s')
else:
    print('Skipped Nash-DQN training')


## Train Locally Linear-Quadratic SRE-DQN


In [ ]:
llq_sre_agents = {}
llq_sre_losses = {}
llq_sre_train_times = {}

if TRAIN_LLQ_SRE:
    for eps in LLQ_SRE_EPS_LIST:
        eps_dir = LLQ_SRE_MODEL_DIRS[eps]
        eps_dir.mkdir(parents=True, exist_ok=True)

        seed_everything(LLQ_SRE_TRAIN_SEED)
        sre_agent = SreNN(
            **NET_KWARGS,
            eps_reg=SRE_EPS_REG,
            delta_min=SRE_DELTA_MIN,
            gamma=SRE_GAMMA,
        )

        start = time.time()
        trained_agent, train_loss = run_training_loop(
            sim_obj=sim_obj,
            sim_dict=sim_dict,
            max_steps=MAX_STEPS,
            agent=sre_agent,
            make_action_fn=make_llq_sre_action,
            eps_schedule_fn=make_eps_schedule(eps),
            num_sim=NUM_SIM,
            norm_mean=norm_mean,
            norm_std=norm_std,
            rv_min=RV_MIN,
            rv_max=RV_MAX,
            early_stop=EARLY_STOP,
            early_lim=EARLY_LIM,
            mini_batch=MINI_BATCH,
            path=str(eps_dir),
            AN_file_name=str(eps_dir / f'LLQ_SRE_Action_Net_eps_{eps_slug(eps)}'),
            VN_file_name=str(eps_dir / f'LLQ_SRE_Value_Net_eps_{eps_slug(eps)}'),
            checkpoint_metadata={
                'trainer': 'llq_sre_dqn',
                'seed': int(LLQ_SRE_TRAIN_SEED),
                'eps_0': float(eps),
                'eps_decay_horizon': None if SRE_EPS_DECAY is None else int(SRE_EPS_DECAY),
                'eps_reg': float(SRE_EPS_REG),
                'delta_min': float(SRE_DELTA_MIN),
                'gamma': float(SRE_GAMMA),
            },
            desc=f'LLQ SRE-DQN eps={eps:g}',
            **TRAINING_LOG_KWARGS,
        )
        llq_sre_agents[eps] = trained_agent
        llq_sre_losses[eps] = train_loss
        llq_sre_train_times[eps] = time.time() - start
        print(f'LLQ SRE-DQN training time (eps={eps:g}): {llq_sre_train_times[eps]:.1f}s')
else:
    print('Skipped LLQ SRE-DQN training')


## Loss Curves


In [ ]:
plt.figure(figsize=(12, 6))
if nash_loss is not None:
    plt.plot(nash_loss, label='Nash-DQN', alpha=0.8)
for eps, loss in llq_sre_losses.items():
    plt.plot(loss, label=f'LLQ SRE eps={eps:g}', alpha=0.8)
plt.yscale('symlog')
plt.xlabel('Training iteration')
plt.ylabel('Training loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()